# Matrike pričakovanih časov srečanja

## 1. Numerični izračuni matrik pričakovanih časov srečanja

In [17]:
import time
import pandas as pd
import numpy as np
import networkx as nx
from scipy.sparse.linalg import bicgstab

In [18]:
STEVILO_VOZLISC = 6
n = STEVILO_VOZLISC

# 1. Definicija 7 usmerjenih grafov (n = 6)
grafi = {
    "Path": nx.path_graph(n).to_directed(),
    "Ring": nx.cycle_graph(n).to_directed(),
    "Star": nx.star_graph(n-1).to_directed(),
    "Lattice": nx.convert_node_labels_to_integers(nx.grid_2d_graph(3, 2)).to_directed(),
    "Lollipop": nx.lollipop_graph(m=4, n=2).to_directed(),
    "Tadpole": nx.tadpole_graph(m=4, n=2).to_directed(),
}

# Sun graf
G_sun = nx.cycle_graph(3)
for i in range(3):
    G_sun.add_edge(i, i + 3)
grafi["Sun"] = G_sun.to_directed()

In [19]:
# 2. Priprava stohastičnih prehodnih matrik P
def vrni_prehodno_matriko(G):
    A = nx.to_numpy_array(G, dtype=float)
    vsote = A.sum(axis=1, keepdims=True)
    vsote[vsote == 0] = 1.0
    return A / vsote

# 3. Priprava matrike E po navodilih iz izreka
# vec(I_n) v Pythonu pripravimo s sploščenjem enotske matrike I_n po stolpcih (order='F')

def vec_M_bicgstab(P):
    """
    Izračuna matriko M pričakovanih časov srečanja z bicgstab
    """
    n = P.shape[0]
    n2 = n * n
    
    # 1. Kroneckerjev produkt P \otimes P
    P_kron = np.kron(P, P)
    
    # 2. Diagonalna matrika E = diag(1_{n^2} - vec(I_n))
    diag_E = np.ones(n2, dtype=float)
    srecanja_indices = [k * n + k for k in range(n)]
    diag_E[srecanja_indices] = 0.0
    E = np.diag(diag_E)
    
    # 3. Zmnožek (P \otimes P) * E
    PPE = P_kron.dot(E)
    
    # 4. Sistem A = I_{n^2} - (P \otimes P) * E
    I_n2 = np.eye(n2)
    A = I_n2 - PPE
    b = np.ones(n2, dtype=float)
    
    # 5. Reševanje sistema A * vec(M) = 1_{n^2}
    vec_M, info = bicgstab(A, b, rtol=1e-8, maxiter=2000)
    
    if info != 0:
        print(f"Opozorilo: bicgstab ni popolnoma konvergiral (koda: {info}).")
        
    M = vec_M.reshape((n, n))
    return M

def vec_M_linalg_solve(P):
    """
    Izračuna matriko M pričakovanih časov srečanja z linalg.solve
    """
    n = P.shape[0]
    n2 = n * n
    
    # 1. Kroneckerjev produkt P \otimes P
    P_kron = np.kron(P, P)
    
    # 2. Diagonalna matrika E = diag(1_{n^2} - vec(I_n))
    diag_E = np.ones(n2, dtype=float)
    srecanja_indices = [k * n + k for k in range(n)]
    diag_E[srecanja_indices] = 0.0
    E = np.diag(diag_E)
    
    # 3. Zmnožek (P \otimes P) * E
    PPE = P_kron.dot(E)
    
    # 4. Sistem A = I_{n^2} - (P \otimes P) * E
    I_n2 = np.eye(n2)
    A = I_n2 - PPE
    b = np.ones(n2, dtype=float)
    
    # 5. Reševanje sistema A * vec(M) = 1_{n^2}
    vec_M = np.linalg.solve(A, b)
    
    M = vec_M.reshape((n, n))
    return M

def vec_M_lstsq(P):
    """
    Izračuna matriko M pričakovanih časov srečanja z lstsq
    """
    n = P.shape[0]
    n2 = n * n
    
    # 1. Kroneckerjev produkt P \otimes P
    P_kron = np.kron(P, P)
    
    # 2. Diagonalna matrika E = diag(1_{n^2} - vec(I_n))
    diag_E = np.ones(n2, dtype=float)
    srecanja_indices = [k * n + k for k in range(n)]
    diag_E[srecanja_indices] = 0.0
    E = np.diag(diag_E)
    
    # 3. Zmnožek (P \otimes P) * E
    PPE = P_kron.dot(E)
    
    # 4. Sistem A = I_{n^2} - (P \otimes P) * E
    I_n2 = np.eye(n2)
    A = I_n2 - PPE
    b = np.ones(n2, dtype=float)
    
    # 5. Reševanje sistema A * vec(M) = 1_{n^2}
    vec_M = np.linalg.lstsq(A, b, rcond=1e-10)[0]
    
    M = vec_M.reshape((n, n))
    return M

### Numerični izračun z metodo ``bicgstab``

In [20]:

# Izračun matrike M za vsak graf

matrike_M = {}

print("=== TEORETIČNE MATRIKE PRIČAKOVANIH ČASOV SREČANJA (M) z BICGSTAB===\n")

for ime, G_dir in grafi.items():
    P = vrni_prehodno_matriko(G_dir)
    m = P.shape[0]
    start = time.perf_counter()
    M = vec_M_bicgstab(P)
    end = time.perf_counter()
    print(f"Čas izračuna matrike M za graf '{ime}': {end - start:.4f} sekund")
    matrike_M[ime] = M
    
    print(f"Topologija: {ime}")
    df_M = pd.DataFrame(M, index=range(m), columns=range(m))
    print(df_M.round(2))
    print("\n" + "="*55 + "\n")

=== TEORETIČNE MATRIKE PRIČAKOVANIH ČASOV SREČANJA (M) z BICGSTAB===

Opozorilo: bicgstab ni popolnoma konvergiral (koda: 2000).
Čas izračuna matrike M za graf 'Path': 0.0847 sekund
Topologija: Path
    0   1   2   3   4   5
0 NaN NaN NaN NaN NaN NaN
1 NaN NaN NaN NaN NaN NaN
2 NaN NaN NaN NaN NaN NaN
3 NaN NaN NaN NaN NaN NaN
4 NaN NaN NaN NaN NaN NaN
5 NaN NaN NaN NaN NaN NaN


Opozorilo: bicgstab ni popolnoma konvergiral (koda: 2000).


c:\Users\Lara\miniconda3\Lib\site-packages\scipy\sparse\linalg\_isolve\iterative.py:294: RuntimeWarning: overflow encountered in multiply
  x += alpha*phat
c:\Users\Lara\miniconda3\Lib\site-packages\scipy\sparse\linalg\_isolve\iterative.py:272: RuntimeWarning: overflow encountered in multiply
  p *= beta
c:\Users\Lara\miniconda3\Lib\site-packages\scipy\sparse\linalg\_interface.py:828: RuntimeWarning: invalid value encountered in dot
  return self.A.dot(X)
c:\Users\Lara\miniconda3\Lib\site-packages\scipy\sparse\linalg\_isolve\iterative.py:294: RuntimeWarning: overflow encountered in multiply
  x += alpha*phat
c:\Users\Lara\miniconda3\Lib\site-packages\scipy\sparse\linalg\_isolve\iterative.py:294: RuntimeWarning: invalid value encountered in add
  x += alpha*phat
c:\Users\Lara\miniconda3\Lib\site-packages\scipy\sparse\linalg\_isolve\iterative.py:272: RuntimeWarning: overflow encountered in multiply
  p *= beta
c:\Users\Lara\miniconda3\Lib\site-packages\scipy\sparse\linalg\_interface.py:8

Čas izračuna matrike M za graf 'Ring': 0.1556 sekund
Topologija: Ring
    0   1   2   3   4   5
0 NaN NaN NaN NaN NaN NaN
1 NaN NaN NaN NaN NaN NaN
2 NaN NaN NaN NaN NaN NaN
3 NaN NaN NaN NaN NaN NaN
4 NaN NaN NaN NaN NaN NaN
5 NaN NaN NaN NaN NaN NaN


Čas izračuna matrike M za graf 'Star': 0.0042 sekund
Topologija: Star
              0             1             2             3             4  \
0  1.800000e+00 -9.191443e+16 -9.191443e+16 -9.191443e+16 -9.191443e+16   
1 -9.191443e+16  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00   
2 -9.191443e+16  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00   
3 -9.191443e+16  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00   
4 -9.191443e+16  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00   
5 -9.191443e+16  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00   

              5  
0 -9.191443e+16  
1  1.000000e+00  
2  1.000000e+00  
3  1.000000e+00  
4  1.000000e+00  
5  1.000000e+00  


Čas izračuna matrike M

### Numerični izračun z metodo ``linalg.solve``

In [21]:
matrike_M = {}

print("=== TEORETIČNE MATRIKE PRIČAKOVANIH ČASOV SREČANJA (M) Z LINALG.SOLVE ===\n")

for ime, G_dir in grafi.items():
    P = vrni_prehodno_matriko(G_dir)
    m = P.shape[0]
    start = time.perf_counter()
    M = vec_M_linalg_solve(P)
    end = time.perf_counter()
    print(f"Čas izračuna matrike M za graf '{ime}': {end - start:.4f} sekund")
    matrike_M[ime] = M
    
    print(f"Topologija: {ime}")
    df_M = pd.DataFrame(M, index=range(m), columns=range(m))
    print(df_M.round(2))
    print("\n" + "="*55 + "\n")

=== TEORETIČNE MATRIKE PRIČAKOVANIH ČASOV SREČANJA (M) Z LINALG.SOLVE ===

Čas izračuna matrike M za graf 'Path': 0.0010 sekund
Topologija: Path
              0             1             2             3             4  \
0  1.000000e+00 -5.004000e+16  3.330000e+00 -5.004000e+16  6.670000e+00   
1 -5.004000e+16  2.670000e+00 -5.004000e+16  4.670000e+00 -5.004000e+16   
2  3.330000e+00 -5.004000e+16  3.330000e+00 -5.004000e+16  4.670000e+00   
3 -5.004000e+16  4.670000e+00 -5.004000e+16  3.330000e+00 -5.004000e+16   
4  6.670000e+00 -5.004000e+16  4.670000e+00 -5.004000e+16  2.670000e+00   
5 -5.004000e+16  6.670000e+00 -5.004000e+16  3.330000e+00 -5.004000e+16   

              5  
0 -5.004000e+16  
1  6.670000e+00  
2 -5.004000e+16  
3  3.330000e+00  
4 -5.004000e+16  
5  1.000000e+00  


Čas izračuna matrike M za graf 'Ring': 0.0007 sekund
Topologija: Ring
              0             1             2             3             4  \
0  3.000000e+00  8.106479e+16  4.000000e+00  8.106479e+1

### Numerični izračun z metodo ``linalg.lstsq``

In [22]:
matrike_M = {}

print("=== TEORETIČNE MATRIKE PRIČAKOVANIH ČASOV SREČANJA (M) z LINALG.LSTSQ ===\n")

for ime, G_dir in grafi.items():
    P = vrni_prehodno_matriko(G_dir)
    m = P.shape[0]
    start = time.perf_counter()
    M = vec_M_lstsq(P)
    end = time.perf_counter()
    print(f"Čas izračuna matrike M za graf '{ime}': {end - start:.4f} sekund")
    matrike_M[ime] = M
    
    print(f"Topologija: {ime}")
    df_M = pd.DataFrame(M, index=range(m), columns=range(m))
    print(df_M.round(2))
    print("\n" + "="*55 + "\n")

=== TEORETIČNE MATRIKE PRIČAKOVANIH ČASOV SREČANJA (M) z LINALG.LSTSQ ===

Čas izračuna matrike M za graf 'Path': 0.0009 sekund
Topologija: Path
      0     1     2     3     4     5
0  1.00  0.42  3.33  0.10  6.67  0.47
1  0.42  2.67 -0.34  4.67 -0.22  6.67
2  3.33 -0.34  3.33 -0.61  4.67  0.10
3  0.10  4.67 -0.61  3.33 -0.34  3.33
4  6.67 -0.22  4.67 -0.34  2.67  0.42
5  0.47  6.67  0.10  3.33  0.42  1.00


Čas izračuna matrike M za graf 'Ring': 0.0010 sekund
Topologija: Ring
     0    1    2    3    4    5
0  3.0 -0.0  4.0  0.0  4.0  0.0
1 -0.0  3.0  0.0  4.0  0.0  4.0
2  4.0 -0.0  3.0  0.0  4.0 -0.0
3 -0.0  4.0 -0.0  3.0  0.0  4.0
4  4.0 -0.0  4.0  0.0  3.0 -0.0
5 -0.0  4.0  0.0  4.0  0.0  3.0


Čas izračuna matrike M za graf 'Star': 0.0007 sekund
Topologija: Star
     0    1    2    3    4    5
0  1.8  0.0 -0.0  0.0  0.0 -0.0
1 -0.0  1.0  1.0  1.0  1.0  1.0
2 -0.0  1.0  1.0  1.0  1.0  1.0
3 -0.0  1.0  1.0  1.0  1.0  1.0
4  0.0  1.0  1.0  1.0  1.0  1.0
5 -0.0  1.0  1.0  1.0  1.0  1

## 2. Izračun matrik pričakovanih časov srečanja iz rezultatov simulacij

In [23]:

# 1. Naložimo rezultate simulacij iz CSV datoteke
INPUT_CSV = "sinteticni_grafi_rezultati.csv"
df = pd.read_csv(INPUT_CSV)

# 2. Funkcija za ustvarjanje matrike povprečnih korakov za posamezno topologijo
def ustvari_empirično_matriko(df_topologija, n=6):
    # Izračunamo povprečje korakov za vsak par (Zacetno_A, Zacetno_B)
    # Upoštevamo le uspešna srečanja (kjer Koncno ni prazno)
    df_uspesni = df_topologija.dropna(subset=['Koncno'])
    
    povprečja = (
        df_uspesni.groupby(['Zacetno_A', 'Zacetno_B'])['Koraki']
        .mean()
        .reset_index()
    )
    
    # Pripravimo prazno matriko z NaN vrednostmi
    matrika = np.full((n, n), np.nan)
    
    # Napolnimo matriko s povprečji
    for _, row in povprečja.iterrows():
        i = int(row['Zacetno_A'])
        j = int(row['Zacetno_B'])
        matrika[i, j] = row['Koraki']
        
    return matrika

# 3. Zanka čez vse topologije in izpis matrik
simulacijske_matrike = {}

print("=== EMPIRIČNE MATRIKE IZ SIMULACIJ (Povprečni koraki) ===\n")

topologije = df['Topologija'].unique()

for topo in topologije:
    df_topo = df[df['Topologija'] == topo]
    
    # Izračun matrike za trenutno topologijo
    M_empiricna = ustvari_empirično_matriko(df_topo, n=6)
    simulacijske_matrike[topo] = M_empiricna
    
    # Pretvorba v DataFrame za lepši prikaz
    df_M = pd.DataFrame(M_empiricna, index=range(6), columns=range(6))
    
    print(f"Topologija: {topo}")
    # Prikažemo zaokroženo na 2 decimalni mesti (NaN pomeni, da se nista srečala)
    print(df_M.round(2).to_string(na_rep=' NaN'))
    print("\n" + "="*55 + "\n")

=== EMPIRIČNE MATRIKE IZ SIMULACIJ (Povprečni koraki) ===

Topologija: Path
      0     1     2     3     4     5
0  1.00   NaN  3.37   NaN  7.20   NaN
1   NaN  2.48   NaN  4.21   NaN  6.82
2  3.03   NaN  3.10   NaN  4.61   NaN
3   NaN  4.33   NaN  2.87   NaN  3.08
4  6.73   NaN  4.48   NaN  2.63   NaN
5   NaN  6.79   NaN  2.81   NaN  1.00


Topologija: Ring
      0     1     2     3     4     5
0  3.20   NaN  3.92   NaN  4.57   NaN
1   NaN  3.20   NaN  3.64   NaN  4.26
2  4.71   NaN  2.97   NaN  4.47   NaN
3   NaN  3.76   NaN  2.93   NaN  3.96
4  3.91   NaN  4.46   NaN  3.09   NaN
5   NaN  4.29   NaN  4.14   NaN  3.15


Topologija: Star
      0    1    2    3    4    5
0  1.83  NaN  NaN  NaN  NaN  NaN
1   NaN  1.0  1.0  1.0  1.0  1.0
2   NaN  1.0  1.0  1.0  1.0  1.0
3   NaN  1.0  1.0  1.0  1.0  1.0
4   NaN  1.0  1.0  1.0  1.0  1.0
5   NaN  1.0  1.0  1.0  1.0  1.0


Topologija: Lattice
      0     1     2     3     4     5
0  2.12   NaN   NaN  2.87  3.24   NaN
1   NaN  2.67  3.57   NaN